# E2B

<img src="./assets/aliyun-sandbox.svg">

函数计算：https://www.aliyun.com/product/fc

region对照表：https://help.aliyun.com/zh/ecs/user-guide/regions-and-zones

## 安装 E2B 依赖

In [ ]:
!uv add e2b

## 沙箱管理

### 创建沙箱

In [ ]:
from langchain_python.core.config import sandbox_settings

sandbox_settings

In [ ]:
from e2b import AsyncSandbox

# 准备好沙箱配置
basic_config = {
    "timeout": 300, # 从创建开始计时，时间到了自动销毁
    "api_key": sandbox_settings.api_key,
    "api_url": sandbox_settings.api_url,
    "domain": sandbox_settings.domain,
}

sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config,
) 
sandbox.sandbox_id

### 连接已有沙箱

In [ ]:
sandbox = await AsyncSandbox.connect(
    sandbox_id=sandbox.sandbox_id, 
    **basic_config
)
sandbox.sandbox_id

### 手动销毁沙箱

In [ ]:
await sandbox.kill()

### 手动刷新时间

In [ ]:
await sandbox.set_timeout(300) # 从现在起再重新计时300秒后销毁

## 沙箱操作

### 文件读写

In [ ]:
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config
) 

In [ ]:
file_path = "/home/user/workspace/test.txt"
await sandbox.set_timeout(300)
await sandbox.files.write(file_path, "Hello World!")

In [ ]:
await sandbox.set_timeout(300)
await sandbox.files.read(file_path)

### 命令执行

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("node --version")

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("npm init -y", cwd="/home/user/workspace")

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/workspace", timeout=60)

## 挂载 OSS

In [ ]:
import json
metadata_config = {
    "metadata":{
        "fc.sandbox.storage.oss": json.dumps({
            "mountPoints": [
                {
                    "bucketName": sandbox_settings.oss_bucket,
                    "mountDir": "/home/user/workspace",
                    "bucketPath": "/e2b-test/workspace",
                    "endpoint": sandbox_settings.oss_endpoint,
                    "readOnly": False,
                },
                {
                    "bucketName": sandbox_settings.oss_bucket,
                    "mountDir": "/home/user/output",
                    "bucketPath": "/e2b-test/output",
                    "endpoint": sandbox_settings.oss_endpoint,
                    "readOnly": False,
                }
            ]
        }),
        "fc.sandbox.auth.role": sandbox_settings.role_arn,
    }
}

In [ ]:
# 创建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

In [ ]:
# 创建文件
await sandbox.set_timeout(300)
await sandbox.files.write("/home/user/workspace/source.txt", "source")
await sandbox.files.write("/home/user/output/output.txt", "output")

In [ ]:
await sandbox.kill()

In [ ]:
# 新建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/workspace")

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/output")

## 沙箱实战

In [ ]:
# 创建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm create vite@latest my-vue-app -- --template vue --no-interactive", cwd="/home/user/workspace"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm i", cwd="/home/user/workspace/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.files.write("/home/user/workspace/my-vue-app/vite.config.js", """
import vue from '@vitejs/plugin-vue';
import { defineConfig } from 'vite';

// https://vite.dev/config/
export default defineConfig({
  plugins: [vue()],
  base: '',
});
""")

In [ ]:
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm run build", cwd="/home/user/workspace/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "cp -a /home/user/workspace/my-vue-app/dist /home/user/output/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.kill()